# Document pipeline (features + test cases)

**Beyond classic Q&A RAG:** this notebook runs the **end-to-end document workflow** via `qbrain_rag.application.document_pipeline.run_document_pipeline`.

1. **Ingest** the same real JDECo PDF (path from `resolve_default_srs_path`).
2. **Chunk + FAISS** (in memory).
3. **Feature extraction** — all chunks (in order) are sent to the LLM as one context (bounded by `max_context_chars` in `feature_extraction`).
4. **Test cases** — for **each extracted feature**, `similarity_search` pulls top-k chunks and the LLM generates `testCases`.

**Default:** `max_features=None` → the model returns as many features as it finds in context, and **every** feature gets test cases (can be expensive).

**Cost control (optional):** set `MAX_FEATURES = 10` in the next cell, or `skip_test_cases=True` for features only. CLI: `python scripts/run_document_pipeline.py doc.pdf --max-features 10`.

**CLI (all features):** `python scripts/run_document_pipeline.py <path-to-pdf>` — omit `--max-features` unless you want a cap.

**Feature style:** consolidated — roughly **one feature per stable requirement label** when the document provides one, with sub-bullets in `description` / `acceptanceCriteria` (see `qbrain_rag.application.prompts_srs`).


In [1]:
%pip install -q langchain-text-splitters langchain-core langchain-community langchain-openai faiss-cpu pymupdf openai python-dotenv


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import sys
from pathlib import Path

# Run this notebook from the notebooks/ folder (parent = rag_lab).
RAG_LAB = Path("..").resolve()
SRC = RAG_LAB / "src"
sys.path.insert(0, str(SRC))

import json

from qbrain_rag.application.document_pipeline import run_document_pipeline
from qbrain_rag.config.notebook_defaults import resolve_default_srs_path

srs_path = resolve_default_srs_path(RAG_LAB)

# None = extract and process ALL features the LLM returns (full document coverage; many API calls).
# Set to an int (e.g. 10) only to limit cost while experimenting.
# Automated runs: set env NB_MAX_FEATURES=3 (optional) so nbconvert finishes in reasonable time.
_nf = os.environ.get("NB_MAX_FEATURES", "").strip()
MAX_FEATURES = int(_nf) if _nf.isdigit() else None
SKIP_TESTS = os.environ.get("NB_SKIP_TESTS", "").strip() in ("1", "true", "yes")

result = run_document_pipeline(
    srs_path,
    max_features=MAX_FEATURES,
    skip_test_cases=SKIP_TESTS,
)

print(json.dumps(result["metadata"], indent=2))
print("\nFeatures returned:", len(result["features"]))
for feat in result["features"]:
    name = feat.get("name", "?")
    n_tc = len(feat.get("testCases", []))
    print(f"  - {name[:70]!r} ... test cases: {n_tc}")


[doc] Loading: D:\Qbrainpython\QBrain\rag_lab\data\srs\JDECo_SRS.docx[1].pdf


[doc] Text length: 45,175 chars


[doc] Chunks: 30 — building FAISS...


[doc] Extracting features (LLM)...


[doc] Features: 7 total, using 3 for tests.


...

[doc]   → 4 test case(s)


[doc] Tests for feature 3/3: 'Inventory Management'...


[doc]   → 5 test case(s)


[doc] Done.


{
  "total_chunks": 30,
  "chunks_fully_in_context": 30,
  "context_truncated": false,
  "feature_count": 3,
  "test_cases_skipped": false
}

Features returned: 3
  - 'Service Request Management' ... test cases: 5
  - 'Service Request Notifications' ... test cases: 4
  - 'Inventory Management' ... test cases: 5


### Inspect one feature (optional)

Uncomment to pretty-print the first feature and its generated tests.


In [1]:
# import json
# print(json.dumps(result["features"][0], indent=2, ensure_ascii=False))
